In [ ]:
import os, sys, subprocess, 
# import server?

In [ ]:
import socket
import time
import uuid
from typing import List, Dict, Optional


class Process:
    def __init__(
        self,
        identifier: Optional[str] = None,
        public_key: str = "",
        connection_type: str = "local",   # local | external
        protocol: str = "tcp",
        port: Optional[int] = None,
    ):
        # --- Identity ---
        self.node = {
            "id": identifier or str(uuid.uuid4()),
            "role": "client",
            "hostname": socket.gethostname(),
            "ip": self._get_ip(),
        }

        # --- Connection ---
        self.connection = {
            "type": connection_type,
            "protocol": protocol,
            "port": port,
            "session_id": str(uuid.uuid4()),
            "established": False,
            "timestamp": time.time(),
        }

        # --- Crypto ---
        self.crypto = {
            "public_key": public_key,
            "key_id": self._fingerprint(public_key),
            "encryption_alg": None,
            "tunnel_valid": False,
        }

        # --- Validation ---
        self.validation = {
            "test_message": None,
            "encrypted_message": None,
            "response_message": None,
            "integrity_ok": False,
            "round_trip_time_ms": None,
            "last_checked": None,
        }

        # --- Routing ---
        self.route = {
            "origin": self.node["id"],
            "destination": None,
            "hops": [],
            "intermediaries": [],
        }

        # --- Delivery ---
        self.delivery = {
            "status": "pending",
            "failure_reason": None,
            "last_attempt": None,
            "retry_count": 0,
        }

        # --- Security ---
        self.security = {
            "zeroization_required": False,
            "zeroization_trigger": None,
            "credentials_stored": True,
            "zeroized": False,
            "zeroization_time": None,
        }

        # --- Audit লগ ---
        self.audit = {
            "events": []
        }

    # -------------------------
    # Internal helpers
    # -------------------------
    def _get_ip(self) -> str:
        try:
            return socket.gethostbyname(socket.gethostname())
        except Exception:
            return "0.0.0.0"

    def _fingerprint(self, key: str) -> str:
        if not key:
            return ""
        return str(hash(key))

    def _log(self, event: str, details: str = ""):
        self.audit["events"].append({
            "timestamp": time.time(),
            "event": event,
            "details": details,
        })

    # -------------------------
    # Public methods
    # -------------------------
    def add_hop(self, node_id: str):
        self.route["hops"].append(node_id)
        self._log("hop_added", node_id)

    def add_intermediary(self, node_id: str, node_type: str):
        self.route["intermediaries"].append({
            "node_id": node_id,
            "type": node_type,
            "validated": False,
            "timestamp": time.time(),
        })
        self._log("intermediary_added", f"{node_id}:{node_type}")

    def validate_intermediary(self, node_id: str):
        for node in self.route["intermediaries"]:
            if node["node_id"] == node_id:
                node["validated"] = True
                node["timestamp"] = time.time()
                self._log("intermediary_validated", node_id)

    def record_validation(self, sent: str, encrypted: str, response: str, integrity_ok: bool, rtt_ms: float):
        self.validation.update({
            "test_message": sent,
            "encrypted_message": encrypted,
            "response_message": response,
            "integrity_ok": integrity_ok,
            "round_trip_time_ms": rtt_ms,
            "last_checked": time.time(),
        })

        self.crypto["tunnel_valid"] = integrity_ok
        self._log("tunnel_validation", f"integrity={integrity_ok}")

    def mark_delivery(self, success: bool, reason: str = ""):
        self.delivery["status"] = "success" if success else "failed"
        self.delivery["failure_reason"] = reason
        self.delivery["last_attempt"] = time.time()

        if not success:
            self.delivery["retry_count"] += 1
            self.security["zeroization_required"] = True
            self.security["zeroization_trigger"] = "delivery_failure"

        self._log("delivery_status", self.delivery["status"])

    def zeroize(self):
        if not self.security["zeroization_required"]:
            return

        # wipe sensitive material
        self.crypto["public_key"] = None
        self.crypto["key_id"] = None

        self.security.update({
            "credentials_stored": False,
            "zeroized": True,
            "zeroization_time": time.time(),
        })

        self._log("zeroization", "credentials wiped")